# Example: The Insurance Problem
A decision-maker with wealth $W$ faces a possible accident and must decide whether to buy insurance. This example makes the decision two ways, maximize expected wealth versus maximize expected utility, and then prices the risk with the certainty equivalent, the risk premium, and the largest premium a risk-averse agent will pay.

The choice is between two scenarios:
* __Scenario A (no insurance):__ with probability $p$ an accident causes damage $d$, leaving wealth $W-d$; with probability $1-p$ nothing happens and wealth stays $W$.
* __Scenario B (insurance):__ the agent pays a premium $r$ that covers all damage, leaving wealth $W-r$ for certain.

> __Learning Objectives.__
>
> At the end of this activity, students will be able to:
> * __Expected value vs. expected utility:__ Show that maximizing expected wealth and maximizing expected utility can give opposite insurance decisions for a risk-averse agent.
> * __Certainty equivalent and risk premium:__ Compute the certainty equivalent $\mathrm{CE}=U^{-1}(\mathbb{E}[U(W)])$ and the risk premium $\pi=\mathbb{E}[W]-\mathrm{CE}$ for the no-insurance lottery.
> * __Maximum premium:__ Determine the largest premium $r_{\max}=W-\mathrm{CE}$ the agent will pay, and confirm $r_{\max}=p\,d+\pi$.

Let's get started.
___

## Theory
The agent has a utility over wealth $U(w)$. For __Scenario A__ (a lottery), the expected wealth and expected utility are
$$
\mathbb{E}[W] = p\,(W-d) + (1-p)\,W = W - p\,d,\qquad
\mathbb{E}[U(W)] = p\,U(W-d) + (1-p)\,U(W).
$$
For __Scenario B__ (certain), the wealth is $W-r$ with utility $U(W-r)$.

* __Expected-value rule:__ insure if $W-r > \mathbb{E}[W]$, i.e., if the premium is below the expected loss $p\,d$.
* __Expected-utility rule:__ insure if $U(W-r) > \mathbb{E}[U(W)]$.

The __certainty equivalent__ of Scenario A is the certain wealth with the same utility as the lottery,
$$
\mathrm{CE} = U^{-1}\!\left(\mathbb{E}[U(W)]\right),
$$
and the __risk premium__ is the wealth the agent gives up to remove the risk,
$$
\pi = \mathbb{E}[W] - \mathrm{CE}.
$$
For a risk-averse agent $U$ is concave, so $\mathrm{CE} < \mathbb{E}[W]$ and $\pi>0$. The largest premium the agent will pay makes Scenario B indifferent to Scenario A, $U(W-r_{\max})=\mathbb{E}[U(W)]$, which gives
$$
r_{\max} = W - \mathrm{CE} = p\,d + \pi.
$$
___

## Setup, Data, and Prerequisites
This example uses functions defined in the `src` directory and a small set of external packages. The `include(...)` call below runs `Include.jl`, which activates the local project environment, loads the packages, and includes our code. The first run may take a few minutes while packages are installed and precompiled.

In [ ]:
include(joinpath(@__DIR__, "Include.jl"));

## Parameters
We use a power utility $U(w)=w^{\tau}$ for $w\geq 0$, with $\tau=0.8<1$, so the agent is risk averse. The asset is worth $W$; an accident (probability $p\in(0,1)$) causes damage $d=\beta W$; the insurance premium is $r=\alpha d$, with $\alpha,\beta\in(0,1)$.

In [ ]:
W = 12000.0;   # wealth / asset value (USD)
p = 0.10;      # probability of an accident
β = 0.90;      # damage as a fraction of W
α = 0.11;      # premium as a fraction of the damage
τ = 0.80;      # power-utility exponent (< 1 => risk averse)

d = β*W;       # damage in an accident (USD)
r = α*d;       # insurance premium (USD)

U = build(MyPowerUtilityFunction, (τ = τ,));   # wealth utility

## Decision 1: Expected Value
Scenario A is a lottery with outcomes $\{W-d,\,W\}$ and probabilities $\{p,\,1-p\}$. We compare its expected wealth with the certain wealth of Scenario B. The next cell computes `wA::Vector{Float64}`, `pA::Vector{Float64}`, `EW_A::Float64`, and `W_B::Float64`, which are used in later cells.

In [ ]:
(; wA, pA, EW_A, W_B) = let
    wA = [W - d, W];     # Scenario A outcomes
    pA = [p, 1 - p];     # Scenario A probabilities

    EW_A = dot(pA, wA);  # expected wealth, Scenario A
    W_B  = W - r;        # certain wealth, Scenario B

    println("E[W] (A, no insurance) = $(round(EW_A, digits = 2)) USD");
    println("W-r  (B, insurance)    = $(round(W_B, digits = 2)) USD");
    println(EW_A ≥ W_B ? "Expected value: do NOT insure" : "Expected value: insure")

    (wA=wA, pA=pA, EW_A=EW_A, W_B=W_B)
end;

## Decision 2: Expected Utility
A risk-averse agent ranks the scenarios by expected utility, not expected wealth. The next cell computes `EU_A::Float64`, which is used in a later cell.

In [ ]:
EU_A = let
    EU_A = expected_utility(U, wA, pA);   # expected utility, Scenario A
    U_B  = U(W_B);                        # utility of the certain wealth, Scenario B

    println("E[U(W)] (A, no insurance) = $(round(EU_A, digits = 3))");
    println("U(W-r)  (B, insurance)    = $(round(U_B, digits = 3))");
    println(U_B ≥ EU_A ? "Expected utility: INSURE" : "Expected utility: do not insure")

    EU_A
end;

### Cross-check by Simulation
The expectations are averages over outcomes. We confirm $\mathbb{E}[W]$ and $\mathbb{E}[U(W)]$ for Scenario A by sampling the accident with a Bernoulli draw.

In [ ]:
let
    Random.seed!(42);
    N = 200_000;
    samples = [ rand() < p ? W - d : W for _ ∈ 1:N ];   # Bernoulli draws of Scenario-A wealth

    println("simulated E[W]    = $(round(sum(samples)/N, digits = 2))   (analytic $(round(EW_A, digits = 2)))");
    println("simulated E[U(W)] = $(round(sum(U, samples)/N, digits = 3))   (analytic $(round(EU_A, digits = 3)))")
end;

## Certainty Equivalent and Risk Premium
We now price the risk of Scenario A. The certainty equivalent is the certain wealth with the same utility as the lottery, and the risk premium is the gap between expected wealth and the certainty equivalent. The next cell computes `CE::Float64` and `risk_premium::Float64`, which are used in a later cell.

In [ ]:
(CE, risk_premium) = let
    CE = certainty_equivalent(U, wA, pA);   # CE = U^{-1}(E[U(W)])
    risk_premium = EW_A - CE;                # π = E[W] - CE
    r_max = W - CE;                          # largest premium the agent will pay

    println("certainty equivalent CE = $(round(CE, digits = 2)) USD");
    println("risk premium π          = $(round(risk_premium, digits = 2)) USD");
    println("max premium r_max = W-CE = $(round(r_max, digits = 2)) USD");
    println("check: r_max ≈ p*d + π   => $(isapprox(r_max, p*d + risk_premium, atol = 1e-6))");
    println("offered premium r = $(round(r, digits = 2)) USD => ",
        r ≤ r_max ? "worth insuring (r ≤ r_max)" : "not worth it (r > r_max)")

    (CE, risk_premium)
end;

## Visualize
We plot the concave utility $U(w)$, the two outcome points, and the chord connecting them. The height of the chord at the expected wealth $\mathbb{E}[W]$ is the expected utility $\mathbb{E}[U(W)]$. The certainty equivalent is where a horizontal line at that height meets the curve, and the risk premium is the horizontal gap $\mathbb{E}[W]-\mathrm{CE}$.

In [ ]:
let
    wgrid = range(0.8*(W - d), stop = 1.05*W, length = 300) |> collect;
    Ucurve = U.(wgrid);

    w_pts = [W - d, W];            # outcome wealth levels
    U_pts = [U(W - d), U(W)];      # utilities of the outcomes

    plot(wgrid, Ucurve, label = "U(w) = w^τ", lw = 3, c = colors[1],
        bg = "floralwhite", background_color_outside = "white", framestyle = :box, fg_legend = :transparent);
    plot!(w_pts, U_pts, label = "chord (expected utility)", lw = 2, ls = :dash, c = colors[3]);
    scatter!(w_pts, U_pts, label = "outcomes", ms = 5, c = "white", mec = colors[1]);

    # expected-utility point on the chord at E[W], and the certainty equivalent on the curve -
    scatter!([EW_A], [EU_A], label = "E[W]=$(round(Int, EW_A)), E[U]=$(round(Int, EU_A))", ms = 6, c = colors[3]);
    scatter!([CE], [EU_A], label = "CE=$(round(Int, CE))", ms = 6, c = colors[4]);
    plot!([CE, EW_A], [EU_A, EU_A], label = "risk premium π=$(round(Int, risk_premium))", lw = 2, ls = :dot, c = colors[4]);

    xlabel!("Wealth w (USD)");
    ylabel!("Utility U(w)");
    current()
end

## What should we expect?
The two decision rules can disagree. Ranking by __expected wealth__ favors insurance only when the premium is below the expected loss $p\,d$; here the premium carries a markup, so the expected-value rule says do not insure. Ranking by __expected utility__ can still favor insurance, because a risk-averse agent has a concave utility, so the certainty equivalent of the risky scenario is below its expected wealth, $\mathrm{CE}<\mathbb{E}[W]$.

The gap between the two is the risk premium $\pi=\mathbb{E}[W]-\mathrm{CE}$, the wealth the agent gives up to remove the risk. The agent insures whenever the premium is at most $r_{\max}=W-\mathrm{CE}=p\,d+\pi$: the actuarially fair premium plus the risk premium. A more risk-averse agent (smaller $\tau$) has a larger risk premium and will pay more.
___

## Key Takeaways

> __What did we learn from this activity?__
>
> * **Expected value and expected utility can give opposite decisions** — a premium above the expected loss $p\,d$ is rejected under the expected-value rule but accepted by a risk-averse agent under the expected-utility rule.
> * **The certainty equivalent prices the risk** — $\mathrm{CE}=U^{-1}(\mathbb{E}[U(W)])$ is below the expected wealth for a concave utility, and the risk premium is $\pi=\mathbb{E}[W]-\mathrm{CE}$.
> * **The risk premium sets the maximum premium** — the agent insures whenever the premium is at most $r_{\max}=W-\mathrm{CE}=p\,d+\pi$.

Try changing the risk-aversion exponent `τ`, the accident probability `p`, or the premium fraction `α` in the parameters and re-running to see when insurance becomes worthwhile.
___

### Additional Resources
* von Neumann, J., & Morgenstern, O. (1944). _Theory of Games and Economic Behavior_. Princeton University Press.
* Pratt, J. W. (1964). Risk aversion in the small and in the large. _Econometrica_, 32(1/2), 122–136.
* Mas-Colell, A., Whinston, M. D., & Green, J. R. (1995). _Microeconomic Theory_. Oxford University Press.